# **SAD2 Lab3: Modeling and simulating Markov Chains**
*Justyna Król*

### **Introduction**

A Markov chain is a way to describe a system that moves between different situations called "states", where the chain assumes the probability of being in a particular state at the next step depends solely on the current state. This key property of ignoring historical states is called the Markov Property or memorylessness and is written as

$$
P(X_{t+1} = j \mid X_t, X_{t-1}, \dots, X_0) = P(X_{t+1} = j \mid X_t = i).
$$

A discrete, time-homogeneous Markov chain on states $S = {1, \dots, N}$ is defined by an $N \times N$ transition matrix $P$. The transition matrix encodes the probabilities of moving from each state to every other state. Each row sums to 1 and represents the conditional distribution of the next state given the current state:

$$
P_{ij} = P(X_{t+1}=j \mid X_t=i).
$$


A stationary distribution $\pi$, which represents the long-term, steady-state probabilities of being in each state, satisfies

$$
\pi P = \pi, \qquad \sum_i \pi_i = 1.
$$

If the chain is ergodic (irreducible and aperiodic), then regardless of the initial state distribution, the empirical frequencies of states converge to $\pi$.



### **Task 1:**
In this task we will implement the core components of a Markov chain. We will:
- Generate a random transition probability matrix $P$.
- Simulate a long state sequence using $P$.
- Estimate the empirical (sample) stationary distribution from the simulated sequence.
- Compute the analytical stationary distribution by solving $\pi P = \pi$.
- Compare and visualize both distributions.
- Plot the evolution of state probabilities over time to observe convergence.


In [ ]:
import torch
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

def set_seed(seed):
  torch.manual_seed(seed)
  np.random.seed(seed)
  random.seed(seed)

Step 1: Define a random transition matrix.

You may either specify your own transition probabilities manually or generate them randomly.
Ensure that each row of the matrix represents a valid probability distribution - all entries must be non-negative and each row sums to 1.

In [ ]:
set_seed(42)
num_states = 5
P = torch.rand(num_states, num_states)
P = P / P.sum(axis=1, keepdim=True)
P

Step 2: Write a function `simulate_markov_chain(P, initial_state, num_steps)` that simulates the evolution of a Markov chain.

The function should:

- Start from `initial_state`.
- At each step, sample the next state according to the probability distribution defined by the transition matrix `P` (Hint: use `torch.multinomial`)
- Append the sampled state to a tensor or list.
- Return the full sequence of states after `num_steps` transitions.

This sequence can then be used to estimate the empirical stationary distribution and analyze the chain’s long-term behavior.



In [ ]:
def simulate_markov_chain(P, initial_state, num_steps):
    # dostajemy wyniki, w każdym wyniku musimy otrzymać jakieś dane 
    states = torch.zeros(num_steps, dtype=torch.long) 
    states[0] = initial_state
    for t in range(1, num_steps):
        states[t] = torch.multinomial(P[states[t-1]], 1, replacement=True).item()

    return states

# Example simulation
initial_state = 0
n_steps = 1000 # 100_000 daję dopiero dobrą aproksymacje
new_sequence = simulate_markov_chain(P, initial_state, n_steps)
print("First 20 states of simulation:", new_sequence[:20])

Step 3: Estimate the stationary distribution and compare it with the analytical stationary distribution.

Use the simulated state sequence to compute the *empirical stationary distribution*, given by the long-run relative frequency of each state:
$$
\hat{\pi}_i = \frac{1}{T} \sum_{t=1}^T \mathbf{1}_{\{X_t = i\}}.
$$
Then compute the *analytical stationary distribution* $\pi$ by solving the equation $\pi P = \pi$ under the constraint $\sum_i \pi_i = 1$.  
Compare the empirical and analytical results to verify that the simulation outcomes converge to the theoretical predictions as $T$ increases.

In [ ]:
def estimate_stationary_distribution(sequence):
  # wyjściowy - problem z outputem
  # T = sequence.shape[0]
  # pd_sequene = pd.DataFrame(sequence, dtype=pd.Int64Dtype)
  # return pd.DataFrame(pd_sequene.value_counts() / T)
  # poprawiony:
  pd_sequene = pd.Series(sequence, dtype=pd.Int16Dtype)
  freq = pd_sequene.value_counts(normalize=True).sort_index()
  return pd.DataFrame({'state': freq.index, 'empirical_pi': freq.values}).set_index('state')

# Analytical stationary distribution: solve πP = π  <=> (P^T - I) π^T = 0
def analytical_stationary_distribution(P):
  num_states = P.shape[0]
  A = P.T - torch.eye(num_states)
  # Append constraint sum(pi)=1
  A = torch.cat([A, torch.ones(1, num_states)], dim=0)
  b = torch.zeros(num_states + 1)
  b[-1] = 1.0
  # Solve least squares
  analytical_pi = torch.linalg.lstsq(A, b.unsqueeze(1)).solution[:num_states].squeeze()
  analytical_pi = analytical_pi / analytical_pi.sum()  # normalize
  return pd.DataFrame({'state': range(num_states), 'analytical_pi': analytical_pi.numpy()}).set_index('state')

empirical_pi = estimate_stationary_distribution(new_sequence)
analytical_pi = analytical_stationary_distribution(P)
print("Empirical stationary distribution:", empirical_pi)
print("Analytical stationary distribution:", analytical_pi)

Step 4: Create a bar plot comparing the empirical stationary distribution $\hat{\pi}$ and the analytical stationary distribution $\pi$.

In [ ]:
df_pi = empirical_pi.join(analytical_pi, 'state', how='outer')
df_pi

In [ ]:
df_pi.plot(kind='bar')
plt.title('Markov chain stationary distribution')

In [ ]:
# TODO

Step 5: Observe how the probabilities of being in each state evolve as the Markov chain progresses and how they converge to the stationary distribution. For each time step $t$, compute the running frequencies of each state $i$:
$$
\hat{\pi}_i(t) = \frac{1}{t} \sum_{s=1}^{t} \mathbf{1}_{\{X_s = i\}},
$$

Use a line plot to visualize $\hat{\pi}_i(t)$ for all states $i$ as a function of time. This plot illustrates the convergence behavior of the chain toward its stationary distribution.

In [ ]:
# simulating
n = 1000
step = 5
probe = simulate_markov_chain(P, 0, n)
result = None
for i in np.arange(1, n, step):
    cdf = estimate_stationary_distribution(probe[:i])
    cdf = cdf.rename(columns={cdf.columns[0]: i})
    if result is None and i > 1:
        result = cdf
    else:
        result = pd.concat([result, cdf], axis=1)
result

In [ ]:
result.transpose().plot()
plt.title('Convergence')
plt.ylabel('probability')

In [ ]:
# Calculate state probabilities over time

...

# Create a line plot showing the running frequencies of each state

...

### **Task 2:** Amino acid sequence modeling

Proteins are composed of sequences of amino acids, and their biological function is determined by the specific arrangement of these amino acids. Understanding the statistical properties of amino acid sequences is essential in bioinformatics, for tasks such as protein family classification, motif discovery, and evolutionary analysis.

In this task, each state of the Markov chain will correspond to one of the 20 standard amino acids, and the transition probabilities will capture how likely one amino acid is to follow another in a protein sequence. We will build the transition matrix $P$ based on a real amino acid sequence.

In [ ]:
sequence = 'MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVLQNMLDLPTWLAINKVFCNKASIRIPWTKLKTHPICLSLDKVIMEMSTCEEPRSPNGPSPIATASGQSEYGFAEKVVEGISVSVNSIVIRIGAKAFNASFELSQLRIYSVNAHWEHGDLRFTRIQDPQRGEVLTFKEINWQMIRIEADATQSSHLEIMCAPVRLITNQSKIRVTLKRRLKDCNVIATKLVLILDDLLWVLTDSQLKAMVQYAKSLSEAIEKSTEQRKSMAPEPTQSSTVVASAQQVKTTQTSNAPDVNDAIVKLFNDFDVKETSHHLVISHLDLHICDDIHAKEKESNRRITGGAMQLSFTQLTIDYYPYHKAGDSCNHWMYFSDATKTKNGWANELLHEFECNVEMLKQAVKDHNVGSPPKSPTHASPQHTQTEKDYPLKGTCRTPSVLSQQSKAKLMSSSVVVRLADFNIYQVSTAEQCRSSPKSMICCNKKSLYLPQEMSAVYIEFTEYYYPDGKDFPIPSPNLYSQLNALQFTVDERSILWLNQFLLDLKQSLNQFMAVYKLNDNSKSDEHVDVRVDGLMLKFVIPSEVKSECHQDQPRAISIQSSEMIATNTRHCPNCRHSDLEALFQDFKDCDFFSKTYTSFPKSCDNFNLLHPIFQRHAHEQDTKMHEIYKGNITPQLNKNTLKTSAATDVWAVYFSQFWIDYEGMKSGKGRPISFVDSFPLSIWICQPTRYAESQKEPQTCNQVSLNTSQSESSDLAGRLKRKKLLKEYYSTESEPLTNGGQKPSSSDTFFRFSPSSSEADIHLLVHVHKHVSMQINHYQYLLLLFLHESLILLSENLRKDVEAVTGSPASQTSICIGILLRSAELALLLHPVDQANTLKSPVSESVSPVVPDYLPTENGDFLSSKRKQISRDINRIRSVTVNHMSDNRSMSVDLSHIPLKDPLLFKSASDTNLQKGISFMDYLSDKHLGKISEDESSGLVYKSGSGEIGSETSDKKDSFYTDSSSILNYREDSNILSFDSDGNQNILSSTLTSKGNETIESIFKAEDLLPEAASLSENLDISKEETPPVRTLKSQSSLSGKPKERCPPNLAPLCVSYKNMKRSSSQMSLDTISLDSMILEEQLLESDGSDSHMFLEKGNKKNSTTNYRGTAESVNAGANLQNYGETSPDAISTNSEGAQENHDDLMSVVVFKITGVNGEIDIRGEDTEICLQVNQVTPDQLGNISLRHYLCNRPVGSDQKAVIHSKSSPEISLRFESGPGAVIHSLLAEKNGFLQCHIENFSTEFLTSSLMNIQHFLEDETVATVMPMKIQVSNTKINLKDDSPRSSTVSLEPAPVTVHIDHLVVERSDDGSFHIRDSHMLNTGNDLKENVKSDSVLLTSGKYDLKKQRSVTQATQTSPGVPWPSQSANFPEFSFDFTREQLMEENESLKQELAKAKMALAEAHLEKDALLHHIKKMTVE'

# Define states
states_amino = sorted(list(set(sequence)))
num_states_amino = len(states_amino)


Step 1: Write a function `get_transition_matrix` that estimates the transition matrix based on a sequence of states.
- Count how often each state is followed by every other state.
- Normalize each row so that the sum of probabilities is 1.

In [ ]:
def get_transition_matrix(sequence):
    # changing sequence to 
    # przeszukujemy całą listę, zbieramy indeksy gdzie występują dany aminokwas  
    # stan i , pytamy sie o stan i+1, i dla każdego stanu chcemy to zliczyć, 
    P_ac = pd.DataFrame(states_amino, columns=['amino acids']).set_index('amino acids')
    for ac in states_amino:
       letters_after_ac = [sequence[i+1] for i, ch in enumerate(sequence) if ch == ac and i+1 < len(sequence)]
       ac_dist = pd.Series(letters_after_ac).value_counts(normalize=True).sort_index()
       P_ac[ac] = ac_dist
    return P_ac.fillna(0)

P_amino = get_transition_matrix(sequence)

In [ ]:
P_amino # to jest macierz przejść 

Step 2: Simulate a new sequence using the transition matrix from step 1 and function `simulate_markov_chain` from Task 1. This time, the initial state should be chosen randomly according to the empirical probabilities of the states in the original sequence.

Calculate empirical and analytical stationary distributions using the functions from Task 1.

In [ ]:
# Simulate new sequence
n_steps = 1000
initial_state_probs = ...
initial_state = ...
simulated_states = simulate_markov_chain(P_amino, initial_state, n_steps)
print("\nSimulated sequence (first 100 characters):")
print(''.join([states_amino[i] for i in simulated_states[:100]]))

empirical_pi_amino = ...
analytical_pi_amino = ...

print("\nEmpirical stationary distribution:", empirical_pi_amino)
print("Analytical stationary distribution:", analytical_pi_amino)

Step 3: Create the same two plots as in task 1.

In [ ]:
# TODO

Step 4: Simulate 10 sequences of the same length. Each time, the initial state should be chosen randomly, as in step 2.
Plot the evolution of observed probabilities of a chosen state in time for each simulated sequence. Comment the outcome.

In [ ]:
n_steps = 1000
n_sequences = 10
state_probs = torch.zeros(n_sequences, n_steps)
for i in range(n_sequences):
    ...